# Experiment Notebook
This note will perform the following :
* Load TypiClust Selections
* Generate random selections
* Train ResNet-18
* Save Results 
* Generate Diagrams 
* Perform statistical analysis - entailing, Paired t-tests, Confidence Intervals and Effect Sizes

In [18]:
import os 
import numpy as np
import torch 
import torchvision
import random 

# Graphical concerns below
import matplotlib.pyplot as plt
import seaborn as sns 
import pandas as pd 
from tqdm import tqdm  # Loading visualiser.

from supervised_training import train_supervised


In [19]:
# Reproducibility measures
torch.manual_seed(0)
np.random.seed(0)
random.seed(0)

# budgets matching the paper 
budgets = [10,20,40,80]
seeds= [0,1,2,3,4] # Statistical Inference

In [20]:
typiclust_selections = {
    B: np.load(f"../TPCRP_Algorithm/results/typiclust_B{B}.npy")
    for B in budgets
}


In [21]:
random_selections = {}

for seed in seeds:
    np.random.seed(seed)
    random_selections[seed] = {
        B: np.random.choice(50000, size=B, replace=False)
        for B in budgets
    }


In [22]:
os.makedirs("results/accuracies", exist_ok=True)
os.makedirs("results/loss_curves", exist_ok=True)
os.makedirs("results/runtimes", exist_ok=True)


In [23]:
results = []

for seed in seeds:
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    for B in budgets:

        # --- TypiClust ---
        typi_idx = typiclust_selections[B]
        acc, loss_curve, runtime = train_supervised(
            selected_indices=typi_idx,
            epochs=50,          # reduced for compute constraints
            batch_size=128,
            lr=0.1,
            momentum=0.9,
            weight_decay=5e-4
        )

        results.append({
            "method": "TypiClust",
            "budget": B,
            "seed": seed,
            "accuracy": acc,
            "runtime": runtime
        })

        # Save loss curve
        np.save(f"results/loss_curves/typiclust_B{B}_seed{seed}.npy", np.array(loss_curve))

        # --- Random ---
        rand_idx = random_selections[seed][B]
        acc, loss_curve, runtime = train_supervised(
            selected_indices=rand_idx,
            epochs=50,
            batch_size=128,
            lr=0.1,
            momentum=0.9,
            weight_decay=5e-4
        )

        results.append({
            "method": "Random",
            "budget": B,
            "seed": seed,
            "accuracy": acc,
            "runtime": runtime
        })

        np.save(f"results/loss_curves/random_B{B}_seed{seed}.npy", np.array(loss_curve))


# CSV Offloading 
df = pd.DataFrame(results)
df.to_csv("results/accuracies/all_results.csv", index=False)
df


URLError: <urlopen error [Errno -3] Temporary failure in name resolution>

In [ ]:
plt.figure(figsize=(8,6))
sns.set(style="whitegrid", font_scale=1.2)

for method in ["TypiClust", "Random"]:
    means = []
    stds = []
    for B in budgets:
        subset = df[(df.method == method) & (df.budget == B)]
        means.append(subset.accuracy.mean())
        stds.append(subset.accuracy.std())

    plt.errorbar(budgets, means, yerr=stds, marker='o', capsize=4, label=method)

plt.xlabel("Label Budget")
plt.ylabel("Test Accuracy")
plt.title("Accuracy vs Budget (CIFAR-10)")
plt.legend()
plt.savefig("results/plots/accuracy_vs_budget.png", dpi=300)
plt.show()


In [ ]:
plt.figure(figsize=(8,6))

gain = []
for B in budgets:
    t = df[(df.method=="TypiClust") & (df.budget==B)].accuracy.mean()
    r = df[(df.method=="Random") & (df.budget==B)].accuracy.mean()
    gain.append(t - r)

plt.plot(budgets, gain, marker='o')
plt.axhline(0, color='black', linestyle='--')
plt.xlabel("Label Budget")
plt.ylabel("Accuracy Gain over Random")
plt.title("TypiClust Improvement over Random")
plt.savefig("results/plots/gain_over_random.png", dpi=300)
plt.show()


In [ ]:
from scipy.stats import ttest_rel

for B in budgets:
    t = df[(df.method=="TypiClust") & (df.budget==B)].accuracy.values
    r = df[(df.method=="Random") & (df.budget==B)].accuracy.values

    stat, p = ttest_rel(t, r)
    print(f"B={B}: p-value = {p:.4e}")


In [ ]:
ssl_loss = np.load("../TPCRP_Algorithm/results/ssl_loss.npy")

plt.figure(figsize=(8,6))
plt.plot(ssl_loss)
plt.xlabel("Epoch")
plt.ylabel("SSL Loss")
plt.title("Self-Supervised Training Loss (NT-Xent)")
plt.savefig("results/plots/ssl_loss_curve.png", dpi=300)
plt.show()


In [ ]:
from sklearn.manifold import TSNE

# Load features from Task 1
features = np.load("../TPCRP_Algorithm/results/features.npy")

tsne = TSNE(n_components=2, perplexity=30)
emb = tsne.fit_transform(features)

plt.figure(figsize=(8,8))
plt.scatter(emb[:,0], emb[:,1], s=2, alpha=0.5)
plt.title("t-SNE of SSL Features")
plt.savefig("results/plots/tsne.png", dpi=300)
plt.show()


In [ ]:
import matplotlib.pyplot as plt

def show_images(indices, title):
    dataset = torchvision.datasets.CIFAR10(root="./data", train=True, download=True)
    plt.figure(figsize=(10,10))
    for i, idx in enumerate(indices[:25]):
        plt.subplot(5,5,i+1)
        plt.imshow(dataset[idx][0])
        plt.axis("off")
    plt.suptitle(title)
    plt.show()

show_images(typiclust_selections[20], "TypiClust Selected Samples (B=20)")
show_images(random_selections[0][20], "Random Selected Samples (B=20)")
